In [1]:
import logging
import os
import sys
import json

import numpy as np
from datasets import load_dataset
import jieba 
from rouge_chinese import Rouge
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import torch

import transformers
from transformers import (
    AutoConfig,
    AutoModel,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    HfArgumentParser,
    Seq2SeqTrainingArguments,
    set_seed,
)
os.environ["PY_ENV"] = "deepseek"
from trainer_seq2seq import Seq2SeqTrainer

from arguments import ModelArguments, DataTrainingArguments

/home/rd-2120/miniconda3/envs/trafficllm-ds/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_files = {"train": "../datasets/changc-qq-2025/sequential_sampling-100/changc-qq-2025_detection_packet_train.json"}
raw_datasets = load_dataset(
        'json',
        data_files=data_files,
        cache_dir="../cache",
        # use_auth_token=True if model_args.use_auth_token else None,
    )
train_dataset = raw_datasets["train"]
train_dataset

Dataset({
    features: ['instruction', 'output'],
    num_rows: 475
})

In [3]:
tokenizer = AutoTokenizer.from_pretrained("../../DeepSeek-R1-Distill-Qwen-7B", trust_remote_code=True)
def preprocess_function_train(examples):
    prompt_column = "instruction"
    response_column = "output"
    max_source_length = 1024
    query, answer = examples[prompt_column], examples[response_column]
    prompt = f"[Round 1]\n\n问：{query}\n\n答："
    instruction = tokenizer(prompt)
    response = tokenizer(answer + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if len(input_ids) > max_source_length:
        input_ids = input_ids[:max_source_length]
        attention_mask = attention_mask[:max_source_length]
        labels = labels[:max_source_length]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [4]:
train_dataset = train_dataset.map(
                preprocess_function_train,
                # batched=True,
                num_proc=10,
                remove_columns=train_dataset.column_names,
                load_from_cache_file=False,
                desc="Running tokenizer on train dataset",
            )
train_dataset

Running tokenizer on train dataset (num_proc=10): 100%|██████████| 475/475 [00:00<00:00, 2458.20 examples/s]


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 475
})